# Notebook 2 — From Signal to Execution

Notebook 1 found a rule: a rolling z-score on the BTC-MINI − ETH-MINI spread, entering at ±2 and flattening below 0.5. That was research — CSV in, chart out, nothing touched the exchange.

This notebook wires the *exact same rule* to the live exchange: poll real prices, maintain the rolling window tick by tick, and place real market orders whenever the signal flips. Nothing here is simulated — it runs against a live matching engine (started locally via `python run.py`, or the real event server if you're at the workshop), and every fill is a real fill against real accounts.

**Setup:** `pip install requests`.

In [ ]:
import statistics
import time
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

DATA_DIR = Path.cwd() / "data"

## The rule, unchanged from notebook 1

| condition | action |
|---|---|
| z > +2 | short BTC-MINI, long ETH-MINI |
| z < −2 | long BTC-MINI, short ETH-MINI |
| ∣z∣ < 0.5 | flatten |
| otherwise | hold |

Notebook 1 used a 300-sample (5-minute) rolling window — appropriate for a 2-hour research pull. This notebook runs live for a few minutes, so the window below is compressed to keep the demo responsive; everything else — thresholds, state machine — is identical. In a real deployment you'd size the window to match whatever horizon you backtested against, not shrink it for demo purposes.

In [ ]:
class Config:
    BASE_URL = "http://127.0.0.1:8000"
    ACCOUNT_ID = "signal_notebook_demo"
    PASSWORD = "notebook2-demo"

    WINDOW = 30          # samples — 30s of 1s polling (notebook 1 used 300 for its 2h pull)
    ENTRY_Z = 2.0
    EXIT_Z = 0.5
    TARGET_QTY = 5       # contracts per leg — well under max_position (15)
    POLL_SECONDS = 1.0
    DURATION_SECONDS = 180  # 3-minute live demo

## Connectivity

Same plumbing as the student notebook: register once (self-serve, active immediately), log in on later runs, `X-API-Key` on every authenticated call.

In [ ]:
def register_or_login(account_id, password):
    r = requests.post(f"{Config.BASE_URL}/register", json={"account_id": account_id, "password": password})
    if r.status_code == 409:
        r = requests.post(f"{Config.BASE_URL}/login", json={"account_id": account_id, "password": password})
    r.raise_for_status()
    return r.json()["api_key"]


def get_products():
    r = requests.get(f"{Config.BASE_URL}/products")
    r.raise_for_status()
    return {p["symbol"]: p for p in r.json()}


def get_account(headers):
    r = requests.get(f"{Config.BASE_URL}/account", headers=headers)
    r.raise_for_status()
    return r.json()


def submit_order(product, side, qty, headers):
    body = {"product": product, "side": side, "type": "market", "qty": qty}
    r = requests.post(f"{Config.BASE_URL}/orders", headers=headers, json=body)
    if not r.ok:
        print("rejected:", r.json().get("detail"))
        return None
    return r.json()


API_KEY = register_or_login(Config.ACCOUNT_ID, Config.PASSWORD)
HEADERS = {"X-API-Key": API_KEY}
print("connected as", Config.ACCOUNT_ID)

## The live signal engine

Same math as notebook 1's loop — rolling mean/std over a fixed window, z-score, hysteresis between entry and exit thresholds — just running one sample at a time instead of over a whole DataFrame.

In [ ]:
class SignalEngine:
    def __init__(self, window, entry_z, exit_z):
        self.window = window
        self.entry_z = entry_z
        self.exit_z = exit_z
        self.spreads = deque(maxlen=window)
        self.position = 0  # -1 short BTC/long ETH, 0 flat, +1 long BTC/short ETH

    def update(self, spread):
        self.spreads.append(spread)
        if len(self.spreads) < self.window:
            return None  # not enough history yet — hold flat

        mean = statistics.fmean(self.spreads)
        std = statistics.pstdev(self.spreads)
        if std == 0:
            return 0.0

        z = (spread - mean) / std
        if self.position == 0:
            if z > self.entry_z:
                self.position = -1
            elif z < -self.entry_z:
                self.position = 1
        elif abs(z) < self.exit_z:
            self.position = 0
        return z

## Translating a position into orders

The signal only says which of three states we want to be in. Reconciliation is the part that makes it real: compare the target position per leg against what the account actually holds right now, and submit a market order for the difference. Sending the diff (not a flat fixed order) means we never double up if a tick is missed and the position is already partway there.

In [ ]:
def target_qtys(position, target_qty):
    return {"BTC-MINI": position * target_qty, "ETH-MINI": -position * target_qty}


def reconcile(target_qty_map, account, headers):
    actions = []
    positions = account["positions"]
    for symbol, target in target_qty_map.items():
        current = positions.get(symbol, {}).get("qty", 0)
        diff = target - current
        if diff == 0:
            continue
        side = "buy" if diff > 0 else "sell"
        resp = submit_order(symbol, side, abs(diff), headers)
        if resp is not None:
            actions.append(f"{side} {abs(diff)}x {symbol}")
    return actions

## Running it live

Every second: pull both index prices, feed the spread into the signal engine, and — only when the target position actually changes — reconcile the account to it. Every tick is logged regardless, so we can chart the whole run afterward exactly like notebook 1's research chart, except this one is live account state, not a CSV.

In [ ]:
engine = SignalEngine(Config.WINDOW, Config.ENTRY_Z, Config.EXIT_Z)
last_position = 0
log = []

start = time.monotonic()
n_ticks = int(Config.DURATION_SECONDS / Config.POLL_SECONDS)
for i in range(n_ticks):
    tick_start = time.monotonic()

    products = get_products()
    btc = products["BTC-MINI"]["index_price"]
    eth = products["ETH-MINI"]["index_price"]
    spread = btc - eth

    z = engine.update(spread)
    position = engine.position

    account = get_account(HEADERS)
    actions = []
    if position != last_position:
        actions = reconcile(target_qtys(position, Config.TARGET_QTY), account, HEADERS)
        last_position = position
        if actions:
            print(f"t={i:>4}s  z={z:.2f}  position={position:+d}  " + ", ".join(actions))

    log.append({
        "timestamp": pd.Timestamp.now(),
        "btc": btc,
        "eth": eth,
        "spread": spread,
        "z": z,
        "signal": position,
        "equity": account["equity"],
        "action": "; ".join(actions),
    })

    elapsed = time.monotonic() - tick_start
    time.sleep(max(0.0, Config.POLL_SECONDS - elapsed))

print(f"\nran {len(log)} ticks over {time.monotonic() - start:.0f}s")

## Flatten at the end

Good hygiene: don't leave a live position resting just because the demo ended mid-signal. Reconcile one last time against a flat target regardless of what the engine currently thinks.

In [ ]:
final_account = get_account(HEADERS)
flatten_actions = reconcile(target_qtys(0, Config.TARGET_QTY), final_account, HEADERS)
print("flattened:", flatten_actions if flatten_actions else "already flat")

final_account = get_account(HEADERS)
print(f"final equity=${final_account['equity']:.2f}  positions={final_account['positions']}")

## What actually happened

Same chart language as notebook 1 — spread line with green/red regime shading — but built from the live log instead of a CSV, plus the equity curve underneath so you can see whether the rule made or lost money over the run.

In [ ]:
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
COLOR_SPREAD = "#1baf7a"
COLOR_EQUITY = "#2a78d6"
LONG_COLOR = "#0ca30c"
SHORT_COLOR = "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRIDLINE, "axes.labelcolor": INK_SECONDARY,
    "text.color": INK_PRIMARY, "xtick.color": INK_SECONDARY, "ytick.color": INK_SECONDARY,
    "grid.color": GRIDLINE, "font.size": 11,
})

run_df = pd.DataFrame(log).set_index("timestamp")

fig, (ax_spread, ax_equity) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, height_ratios=[2, 1])

sig = run_df["signal"].fillna(0).to_numpy()
idx = run_df.index
run_start = 0
for i in range(1, len(sig) + 1):
    if i == len(sig) or sig[i] != sig[run_start]:
        if sig[run_start] != 0:
            color = LONG_COLOR if sig[run_start] > 0 else SHORT_COLOR
            ax_spread.axvspan(idx[run_start], idx[min(i, len(sig) - 1)], color=color, alpha=0.12, linewidth=0)
        run_start = i

ax_spread.plot(run_df.index, run_df["spread"], color=COLOR_SPREAD, linewidth=2)
ax_spread.axhline(0, color=GRIDLINE, linewidth=1)
ax_spread.plot([], [], color=LONG_COLOR, linewidth=8, alpha=0.5, label="long BTC-MINI / short ETH-MINI")
ax_spread.plot([], [], color=SHORT_COLOR, linewidth=8, alpha=0.5, label="short BTC-MINI / long ETH-MINI")
ax_spread.legend(loc="upper left", frameon=False)
ax_spread.set_ylabel("BTC-MINI − ETH-MINI ($)")
ax_spread.set_title("Live run: spread with executed signal regime")
ax_spread.grid(axis="y")

ax_equity.plot(run_df.index, run_df["equity"], color=COLOR_EQUITY, linewidth=2)
ax_equity.set_ylabel("equity ($)")
ax_equity.grid(axis="y")

for spine in ("top", "right"):
    ax_spread.spines[spine].set_visible(False)
    ax_equity.spines[spine].set_visible(False)

fig.tight_layout()
plt.show()

## Talking points

- The signal logic didn't change at all going from notebook 1 to here — same thresholds, same state machine. Only the input (live poll vs. CSV) and the output (real orders vs. a chart) changed. That's deliberate: research and execution should share the exact same decision code, or you're not testing what you'll actually run.
- Reconciliation sends *diffs*, not fixed orders — so a missed tick or a partial fill doesn't compound into a doubled position next time the signal fires.
- This uses **market orders** for simplicity, which means it pays the taker fee and crosses the spread on every rebalance — exactly the transaction cost notebook 1 flagged as unmodeled. Watch the equity curve: every flip costs something before the trade has even had a chance to be right.
- Flattening at the end isn't optional cleanup — leaving a resting position because the demo happened to end mid-signal is a real, unintended bet.
- What's still missing before this is a real strategy: limit orders instead of market (maker rebates, no guaranteed cross), position sizing that scales with conviction instead of a fixed `TARGET_QTY`, and handling for rejected orders (`MAX_POSITION` breaches, rate limits) instead of assuming every call succeeds. Notebook 3 covers how to design a system around those gaps.